In [2]:
# file_path = '/content/drive/My Drive/Master Thesis/stock_return_data.xlsx'
# df = pd.read_excel(file_path)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
from scipy.optimize import minimize
import os
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

# Try to train a first model with torch and custom loss function (later SPO+ loss)

In [4]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data.csv")

In [5]:
return_data

,RIC,date,return
0,POOL.OQ,2000-01-31,-6.521010e-12
1,POOL.OQ,2000-02-29,-4.337349e-02
2,POOL.OQ,2000-03-31,2.342569e-01
3,POOL.OQ,2000-04-28,2.020408e-01
4,POOL.OQ,2000-05-31,-7.979626e-02
...,...,...,...
101723,AVY.N,2023-09-29,-2.604373e-02
101724,AVY.N,2023-10-31,-4.707943e-02
101725,AVY.N,2023-11-30,1.173666e-01
101726,AVY.N,2023-12-29,4.374047e-02


In [6]:
return_data.groupby("RIC").describe()

return                                                              \
         count      mean       std       min       25%       50%       75%   
RIC                                                                          
A.N      289.0  0.009890  0.113212 -0.447458 -0.061024  0.009410  0.070371   
AAPL.OQ  289.0  0.025783  0.112153 -0.577436 -0.039523  0.028987  0.097026   
ABT.N    289.0  0.010300  0.057519 -0.207368 -0.024207  0.012418  0.045617   
ACGL.OQ  289.0  0.015977  0.060538 -0.296067 -0.020703  0.015010  0.045173   
ADBE.OQ  289.0  0.018576  0.112262 -0.334764 -0.044722  0.028672  0.079525   
...        ...       ...       ...       ...       ...       ...       ...   
XOM.N    289.0  0.007848  0.063605 -0.261858 -0.026864  0.003575  0.043759   
XRAY.OQ  289.0  0.007999  0.067672 -0.209412 -0.029487  0.009831  0.046875   
YUM.N    289.0  0.014031  0.072782 -0.258900 -0.029466  0.013321  0.051502   
ZBRA.OQ  289.0  0.012629  0.100059 -0.273250 -0.050383  0.013468  0.069988   
ZION.OQ  289.0  0.006323  0.109302 -0.408732 -0.045000  0.005256  0.049444   

                   
              max  
RIC                
A.N      0.566572  
AAPL.OQ  0.453782  
ABT.N    0.171739  
ACGL.OQ  0.402985  
ADBE.OQ  0.852440  
...           ...  
XOM.N    0.269156  
XRAY.OQ  0.204360  
YUM.N    0.289903  
ZBRA.OQ  0.364994  
ZION.OQ  0.478566  

[352 rows x 8 columns]

In [7]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')
display(return_matrix)

RIC,A.N,AAPL.OQ,ABT.N,ACGL.OQ,ADBE.OQ,ADI.OQ,ADM.N,ADP.OQ,ADSK.OQ,AEE.N,...,WMB.N,WMT.N,WST.N,WY.N,XEL.OQ,XOM.N,XRAY.OQ,YUM.N,ZBRA.OQ,ZION.OQ
date,,,,,,,,,,,,,,,,,,,,,
2000-01-31,-0.143897,0.009119,-0.097065,0.158416,-0.181227,0.005376,-0.035897,-0.119490,-0.092685,-5.725191e-03,...,0.267894,-0.207957,0.005608,-0.195522,-0.012821,0.036462,0.047619,-0.258900,0.011752,0.003960
2000-02-29,0.566572,0.104819,0.003831,0.042735,0.852440,0.679144,-0.139973,-0.081686,0.462168,-7.869482e-02,...,0.079032,-0.110731,-0.048485,-0.105664,-0.087662,-0.092836,0.035354,-0.069869,0.124604,-0.102537
2000-03-31,0.003014,0.184842,0.074427,0.073770,0.091363,0.026274,0.031056,0.109845,0.018182,5.474215e-02,...,0.054239,0.141254,-0.140127,0.110840,0.131673,0.033195,0.109834,0.166667,-0.248826,-0.215548
2000-04-28,-0.147837,-0.086516,0.097600,-0.060115,0.086468,-0.046548,-0.042169,0.115285,-0.155477,1.858586e-01,...,-0.150782,-0.002252,-0.027729,-0.062500,0.116715,-0.001606,0.024229,0.098592,0.140000,-0.003003
2000-05-31,-0.169252,-0.322922,0.058537,-0.025381,-0.069251,0.002441,0.207154,0.020906,-0.030945,-7.407630e-12,...,0.113903,0.040632,-0.038363,-0.064345,0.014327,0.078095,0.055914,-0.141026,-0.157895,0.131392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.076402,-0.088678,-0.058795,0.037080,-0.088390,-0.032262,-0.048928,-0.050382,-0.067721,-4.845823e-02,...,-0.011417,-0.016481,-0.077882,-0.063817,0.010451,0.057469,-0.075219,-0.034318,-0.139922,-0.017183
2023-10-31,-0.073692,-0.002570,-0.018228,0.087442,0.043460,-0.101434,-0.051047,-0.092942,-0.044850,1.175999e-02,...,0.021075,0.021760,-0.151702,-0.064253,0.035827,-0.099762,-0.109778,-0.032656,-0.114573,-0.115792
2023-11-30,0.236335,0.113747,0.103014,-0.034495,0.148386,0.165576,0.036457,0.053616,0.105247,2.483159e-02,...,0.069477,-0.047243,0.102681,0.099338,0.026489,-0.020540,0.044064,0.043727,0.131548,0.168970


In [8]:
# Perform bootstrap
return_array = return_matrix.values

# Number of bootstrap scenarios
num_scenarios = 1000

# Set random seed for reproducibility
np.random.seed(42)

# Sample row indices with replacement
sample_indices = np.random.choice(return_array.shape[0], size=num_scenarios, replace=True)

# Generate bootstrapped scenario matrix
scenario_matrix = return_array[sample_indices, :]

loss_matrix = -scenario_matrix

print(scenario_matrix.shape)

(1000, 352)


In [9]:
# scenario_matrix (S scenarios × N assets)
S, N = scenario_matrix.shape

# Empirical mean from scenarios
mu = scenario_matrix.mean(axis=0)

print(S, N)

1000 352


In [10]:
# Set CVar
alpha = 0.9

def compute_cvar(portfolio_losses, alpha=0.95):
    sorted_losses = np.sort(portfolio_losses)
    k = int(np.ceil(alpha * len(sorted_losses)))
    return sorted_losses[k:].mean()

In [11]:
w = cp.Variable(N)
z = cp.Variable(S)
h = cp.Variable()

cvar_expr = h + (1 / ((1 - alpha) * S)) * cp.sum(z)        # CVaRα(w)

base_constraints = [
    cp.sum(w) == 1,                 # full investment
    w >= 0,
    w <= 0.20,
    z >= 0,
    z >= loss_matrix @ w - h        # z_s ≥ L_s(w) - h
]

prob_min = cp.Problem(cp.Minimize(cvar_expr), base_constraints)
beta_min = prob_min.solve(solver=cp.HIGHS, warm_start=True)

w_max = cp.Variable(N)
prob_max = cp.Problem(cp.Maximize(mu @ w_max),
                      [cp.sum(w_max)==1, w_max>=0, w_max<=0.20])
prob_max.solve()
beta_max = compute_cvar(loss_matrix @ w_max.value, alpha)

print(f"β_min  = {beta_min : .6f}")
print(f"β_max  = {beta_max : .6f}")

β_min  =  0.029533
β_max  =  0.137498


In [12]:
return_matrix

RIC,A.N,AAPL.OQ,ABT.N,ACGL.OQ,ADBE.OQ,ADI.OQ,ADM.N,ADP.OQ,ADSK.OQ,AEE.N,...,WMB.N,WMT.N,WST.N,WY.N,XEL.OQ,XOM.N,XRAY.OQ,YUM.N,ZBRA.OQ,ZION.OQ
date,,,,,,,,,,,,,,,,,,,,,
2000-01-31,-0.143897,0.009119,-0.097065,0.158416,-0.181227,0.005376,-0.035897,-0.119490,-0.092685,-5.725191e-03,...,0.267894,-0.207957,0.005608,-0.195522,-0.012821,0.036462,0.047619,-0.258900,0.011752,0.003960
2000-02-29,0.566572,0.104819,0.003831,0.042735,0.852440,0.679144,-0.139973,-0.081686,0.462168,-7.869482e-02,...,0.079032,-0.110731,-0.048485,-0.105664,-0.087662,-0.092836,0.035354,-0.069869,0.124604,-0.102537
2000-03-31,0.003014,0.184842,0.074427,0.073770,0.091363,0.026274,0.031056,0.109845,0.018182,5.474215e-02,...,0.054239,0.141254,-0.140127,0.110840,0.131673,0.033195,0.109834,0.166667,-0.248826,-0.215548
2000-04-28,-0.147837,-0.086516,0.097600,-0.060115,0.086468,-0.046548,-0.042169,0.115285,-0.155477,1.858586e-01,...,-0.150782,-0.002252,-0.027729,-0.062500,0.116715,-0.001606,0.024229,0.098592,0.140000,-0.003003
2000-05-31,-0.169252,-0.322922,0.058537,-0.025381,-0.069251,0.002441,0.207154,0.020906,-0.030945,-7.407630e-12,...,0.113903,0.040632,-0.038363,-0.064345,0.014327,0.078095,0.055914,-0.141026,-0.157895,0.131392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.076402,-0.088678,-0.058795,0.037080,-0.088390,-0.032262,-0.048928,-0.050382,-0.067721,-4.845823e-02,...,-0.011417,-0.016481,-0.077882,-0.063817,0.010451,0.057469,-0.075219,-0.034318,-0.139922,-0.017183
2023-10-31,-0.073692,-0.002570,-0.018228,0.087442,0.043460,-0.101434,-0.051047,-0.092942,-0.044850,1.175999e-02,...,0.021075,0.021760,-0.151702,-0.064253,0.035827,-0.099762,-0.109778,-0.032656,-0.114573,-0.115792
2023-11-30,0.236335,0.113747,0.103014,-0.034495,0.148386,0.165576,0.036457,0.053616,0.105247,2.483159e-02,...,0.069477,-0.047243,0.102681,0.099338,0.026489,-0.020540,0.044064,0.043727,0.131548,0.168970


## Baue Datensatz für MLP das returns aus letzten 5 lags predicted

was will ich für daten??

Beispiel für EIN Trainingssample (1 prediction):

Input: return distributions of last 3 months

lag3 lag2 lag1

r1   r1   r1

r2   r2   r2

...  ...  ...

r352 r352 r352

Output: return distribution now (predicted or known for training)

r1

r2

...

r352

Abstrahiere auf Trainingsdatensatz:

in X muss in jede Zeile eine Matrix mit so vielen Zeilen wie stocks und Spalten wie lags.

in Y muss in jede Zeile ein Vektor mit so vielen Zeilen wie stocks

In [15]:
return_matrix

RIC,A.N,AAPL.OQ,ABT.N,ACGL.OQ,ADBE.OQ,ADI.OQ,ADM.N,ADP.OQ,ADSK.OQ,AEE.N,...,WMB.N,WMT.N,WST.N,WY.N,XEL.OQ,XOM.N,XRAY.OQ,YUM.N,ZBRA.OQ,ZION.OQ
date,,,,,,,,,,,,,,,,,,,,,
2000-01-31,-0.143897,0.009119,-0.097065,0.158416,-0.181227,0.005376,-0.035897,-0.119490,-0.092685,-5.725191e-03,...,0.267894,-0.207957,0.005608,-0.195522,-0.012821,0.036462,0.047619,-0.258900,0.011752,0.003960
2000-02-29,0.566572,0.104819,0.003831,0.042735,0.852440,0.679144,-0.139973,-0.081686,0.462168,-7.869482e-02,...,0.079032,-0.110731,-0.048485,-0.105664,-0.087662,-0.092836,0.035354,-0.069869,0.124604,-0.102537
2000-03-31,0.003014,0.184842,0.074427,0.073770,0.091363,0.026274,0.031056,0.109845,0.018182,5.474215e-02,...,0.054239,0.141254,-0.140127,0.110840,0.131673,0.033195,0.109834,0.166667,-0.248826,-0.215548
2000-04-28,-0.147837,-0.086516,0.097600,-0.060115,0.086468,-0.046548,-0.042169,0.115285,-0.155477,1.858586e-01,...,-0.150782,-0.002252,-0.027729,-0.062500,0.116715,-0.001606,0.024229,0.098592,0.140000,-0.003003
2000-05-31,-0.169252,-0.322922,0.058537,-0.025381,-0.069251,0.002441,0.207154,0.020906,-0.030945,-7.407630e-12,...,0.113903,0.040632,-0.038363,-0.064345,0.014327,0.078095,0.055914,-0.141026,-0.157895,0.131392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.076402,-0.088678,-0.058795,0.037080,-0.088390,-0.032262,-0.048928,-0.050382,-0.067721,-4.845823e-02,...,-0.011417,-0.016481,-0.077882,-0.063817,0.010451,0.057469,-0.075219,-0.034318,-0.139922,-0.017183
2023-10-31,-0.073692,-0.002570,-0.018228,0.087442,0.043460,-0.101434,-0.051047,-0.092942,-0.044850,1.175999e-02,...,0.021075,0.021760,-0.151702,-0.064253,0.035827,-0.099762,-0.109778,-0.032656,-0.114573,-0.115792
2023-11-30,0.236335,0.113747,0.103014,-0.034495,0.148386,0.165576,0.036457,0.053616,0.105247,2.483159e-02,...,0.069477,-0.047243,0.102681,0.099338,0.026489,-0.020540,0.044064,0.043727,0.131548,0.168970


In [ ]:
X = []
Y = []

max_lag = 5
n_rows = len(return_matrix)

n = max_lag
while n < n_rows:
    # Get X
    X_row = (return_matrix[n - max_lag:n]).values.flatten()
    X.append(X_row)

    # Get Y
    Y_row = (return_matrix.iloc[n]).values.flatten()
    Y.append(Y_row)
    n = n + 1

X = np.array(X)
Y = np.array(Y)

print(X.shape)
print(Y.shape)

(284, 1760)
(284, 352)


In [49]:
train_size = int(0.8 * len(X))

X_train = X[:train_size]
X_test = X[train_size:]

Y_train = Y[:train_size]
Y_test = Y[train_size:]

In [50]:
from sklearn.preprocessing import StandardScaler

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

Y_train = y_scaler.fit_transform(Y_train)
Y_test = y_scaler.transform(Y_test)

In [51]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

In [68]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

batch_size = 16

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [57]:
X_train[0].shape

(1760,)

In [76]:
class ReturnMLP(nn.Module):

    def __init__(self, input_dim, output_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            # nn.Dropout(0.1),

            nn.Linear(1024, 512),
            nn.ReLU(),
            # nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.ReLU(),
            # nn.Dropout(0.1),

            nn.Linear(256, output_dim)
        )

    def forward(self, x):

        return self.network(x)

In [77]:
input_dim = X_train.shape[1]
output_dim = Y_train.shape[1]

model = ReturnMLP(input_dim, output_dim)

print(model)

ReturnMLP(
  (network): Sequential(
    (0): Linear(in_features=1760, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=352, bias=True)
  )
)


In [78]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [79]:
n_epochs = 50

for epoch in range(n_epochs):

    model.train()

    epoch_loss = 0

    for X_batch, Y_batch in train_loader:

        # forward
        predictions = model(X_batch)

        loss = criterion(predictions, Y_batch)

        # backward
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)

    print(f"Epoch {epoch+1}: {avg_loss:.6f}")

Epoch 1: 1.002022
Epoch 2: 0.970499
Epoch 3: 0.927328
Epoch 4: 0.812790
Epoch 5: 0.792808
Epoch 6: 0.712266
Epoch 7: 0.695290
Epoch 8: 0.646166
Epoch 9: 0.587888
Epoch 10: 0.559103
Epoch 11: 0.519609
Epoch 12: 0.510336
Epoch 13: 0.470923
Epoch 14: 0.428622
Epoch 15: 0.400709
Epoch 16: 0.363931
Epoch 17: 0.350234
Epoch 18: 0.422822
Epoch 19: 0.354751
Epoch 20: 0.327893
Epoch 21: 0.318744
Epoch 22: 0.292754
Epoch 23: 0.293685
Epoch 24: 0.271736
Epoch 25: 0.260994
Epoch 26: 0.309131
Epoch 27: 0.254929
Epoch 28: 0.231055
Epoch 29: 0.226492
Epoch 30: 0.220847
Epoch 31: 0.212213
Epoch 32: 0.195718
Epoch 33: 0.191808
Epoch 34: 0.194410
Epoch 35: 0.184143
Epoch 36: 0.187117
Epoch 37: 0.183517
Epoch 38: 0.160106
Epoch 39: 0.181296
Epoch 40: 0.158788
Epoch 41: 0.173919
Epoch 42: 0.169142
Epoch 43: 0.168120
Epoch 44: 0.157781
Epoch 45: 0.143380
Epoch 46: 0.145824
Epoch 47: 0.139320
Epoch 48: 0.158280
Epoch 49: 0.155664
Epoch 50: 0.146665


In [80]:
model.eval()

with torch.no_grad():

    preds_scaled = model(X_test_tensor).numpy()

preds = y_scaler.inverse_transform(preds_scaled)

In [81]:
preds = y_scaler.inverse_transform(preds_scaled)
Y_true = y_scaler.inverse_transform(Y_test_tensor.numpy())

In [82]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(Y_true, preds)
mae = mean_absolute_error(Y_true, preds)
r2  = r2_score(Y_true, preds)

print("GLOBAL STATS")
print("MSE:", mse)
print("MAE:", mae)
print("R2 :", r2)

GLOBAL STATS
MSE: 0.010344273410737514
MAE: 0.07487663626670837
R2 : -0.18001104891300201


In [67]:
asset_mse = np.mean((Y_true - preds)**2, axis=0)
asset_mae = np.mean(np.abs(Y_true - preds), axis=0)

best_assets = np.argsort(asset_mse)[:10]
worst_assets = np.argsort(asset_mse)[-10:]

print("Best 10 assets (lowest MSE):", best_assets)
print("Worst 10 assets (highest MSE):", worst_assets)

Best 10 assets (lowest MSE): [246  68 179 139 182 248  80 188 287 157]
Worst 10 assets (highest MSE): [241 145  40 240 266 103  62 116 216  27]
